## Week Ten - Assignment: Document Classification
 
#### Instructions
###### It can be useful to be able to classify new "test" documents using already classified "training" documents.  A common example is using a corpus of labeled spam and ham (non-spam) e-mails to predict whether or not a new document is spam.  Here is one example of such data:  UCI Machine Learning Repository: Spambase Data Set

###### For this project, you can either use the above dataset to predict the class of new documents (either withheld from the training dataset or from another source such as your own spam folder).

###### For more adventurous students, you are welcome (encouraged!) to come up a different set of documents (including scraped web pages!?) that have already been classified (e.g. tagged), then analyze these documents to predict how new documents should be classified.

### Using the assigned data set found at https://archive.ics.uci.edu/dataset/94/spambase

In [1]:
from ucimlrepo import fetch_ucirepo 
# fetch dataset 
spambase = fetch_ucirepo(id=94) 
# data (as pandas dataframes) 
X = spambase.data.features 
y = spambase.data.targets 
# metadata 
print(spambase.metadata) 
# variable information 
print(spambase.variables) 

{'uci_id': 94, 'name': 'Spambase', 'repository_url': 'https://archive.ics.uci.edu/dataset/94/spambase', 'data_url': 'https://archive.ics.uci.edu/static/public/94/data.csv', 'abstract': 'Classifying Email as Spam or Non-Spam', 'area': 'Computer Science', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 4601, 'num_features': 57, 'feature_types': ['Integer', 'Real'], 'demographics': [], 'target_col': ['Class'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 1999, 'last_updated': 'Mon Aug 28 2023', 'dataset_doi': '10.24432/C53G6X', 'creators': ['Mark Hopkins', 'Erik Reeber', 'George Forman', 'Jaap Suermondt'], 'intro_paper': None, 'additional_info': {'summary': 'The "spam" concept is diverse: advertisements for products/web sites, make money fast schemes, chain letters, pornography...\n\nThe classification task for this dataset is to determine whether a given email is spam or not.\n\t\nOur collecti

Making use of the UCI Spambase dataset to build document classification models to determine/predict if email is spam or not. 

I use several different supervised data models to see which performs best on the classification. 

The data is split into X and y, which are the input features and labels, respectively. Each row is the equivalent to one email document. THe features are informational dimensions like word frequency, character frequency, and capital letter details etc. The label is just whether the email is spam or not.

### Pulling in the rest of what is needed beyond the dataset

In [20]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier


#### Basic data exploration

In [29]:
print("Feature shape: ", X.shape)
print("Target shape: ", y.shape)
# print(X.head())

# print(y.head())

print("Class counts: ",y.value_counts())

print("Missing values in X: ", X.isnull().sum().sum())

Feature shape:  (4601, 57)
Target shape:  (4601, 1)
Class counts:  Class
0        2788
1        1813
Name: count, dtype: int64
Missing values in X:  0


In [15]:
### breaking the data up into training and test sets for model development  
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [16]:
## Scaling the datafor the non-decision tree models. will benefit logistic regression modeling. 
## Scaling after the splits to avoid data leakage. 
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [17]:
## Using the scaled training sets to make the model.
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_scaled, y_train.values.ravel())

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [18]:
# Use the trained model to predict labels for the test set.
# Then looking at classification report and a confusinon matrix 
y_pred_log = log_model.predict(X_test_scaled)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_log))
print("\nClassification Report:\n", classification_report(y_test, y_pred_log))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_log))

Logistic Regression Accuracy: 0.9196525515743756

Classification Report:
               precision    recall  f1-score   support

           0       0.91      0.95      0.93       531
           1       0.93      0.87      0.90       390

    accuracy                           0.92       921
   macro avg       0.92      0.91      0.92       921
weighted avg       0.92      0.92      0.92       921


Confusion Matrix:
 [[506  25]
 [ 49 341]]


Logistic Regression Results:
- Accuracy of 0.9197.
- precision for spam: 0.93
- recall for spam: 0.87
- 25 false positives
- 49 false negatives

In [21]:
### Naive Bayes modeling 
nb_model = GaussianNB()
nb_model.fit(X_train, y_train.values.ravel())

y_pred_nb = nb_model.predict(X_test)

print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred_nb))
print("\nClassification Report:\n", classification_report(y_test, y_pred_nb))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_nb))

Naive Bayes Accuracy: 0.8208469055374593

Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.73      0.82       531
           1       0.72      0.95      0.82       390

    accuracy                           0.82       921
   macro avg       0.83      0.84      0.82       921
weighted avg       0.85      0.82      0.82       921


Confusion Matrix:
 [[387 144]
 [ 21 369]]


Naive Bayes: 
- accuracy of 0.8208
- Recall for spam was .95
- Precision for spam was 0.72
- 144 false positives
- 21 false negatives

This performed worse than the LR first model in precision and false positives. Did worse than the Log Reg. model so not moving this foward for model by model comparison.

In [22]:
# Decision Tree (dontr need scale)
dt_model = DecisionTreeClassifier(random_state=42, max_depth=5)
dt_model.fit(X_train, y_train)

y_pred_dt = dt_model.predict(X_test)

print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred_dt))
print("\nClassification Report:\n", classification_report(y_test, y_pred_dt))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_dt))

Decision Tree Accuracy: 0.9001085776330076

Classification Report:
               precision    recall  f1-score   support

           0       0.87      0.97      0.92       531
           1       0.95      0.81      0.87       390

    accuracy                           0.90       921
   macro avg       0.91      0.89      0.90       921
weighted avg       0.90      0.90      0.90       921


Confusion Matrix:
 [[513  18]
 [ 74 316]]


Decision Tree:
- accuracy of 0.90
- precision of 0.95
- recall of 0.81
- 18 false positives
- 74 false negatives

Has lower accuracy than the Log. Reg. model, higher precision. Lower recall. This had lower false positives but higher false negatives.

In [23]:
## Trying Random Forest tree-type modelz, dont need to scale the numbers 
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train.values.ravel())

y_pred_rf = rf_model.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print("\nClassification Report:\n", classification_report(y_test, y_pred_rf))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))

Random Forest Accuracy: 0.9554831704668838

Classification Report:
               precision    recall  f1-score   support

           0       0.94      0.98      0.96       531
           1       0.98      0.92      0.95       390

    accuracy                           0.96       921
   macro avg       0.96      0.95      0.95       921
weighted avg       0.96      0.96      0.96       921


Confusion Matrix:
 [[522   9]
 [ 32 358]]


Random Forest: 
- accuracy of 0.9555
- precision: 0.98
- recall: 0.92
- 9 false positives
- 32 false negatives

Compared to the Log. Reg. Model, which was the best of the first three, this model did better in accuracy Precision and recall. Additionally, it had better numers for false positives and negatives.
    
This model performed the best.

In [25]:
# Compareall used model results
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Naive Bayes", "Decision Tree", "Random Forest"],
    "Accuracy": [accuracy_score(y_test, y_pred_log),accuracy_score(y_test, y_pred_nb),
                 accuracy_score(y_test, y_pred_dt),accuracy_score(y_test, y_pred_rf)]})

print(results.sort_values(by="Accuracy", ascending=False))

                 Model  Accuracy
3        Random Forest  0.955483
0  Logistic Regression  0.919653
2        Decision Tree  0.900109
1          Naive Bayes  0.820847


### Summary 

This project used the UCI Spambase dataset to build and compare multiple supervised classification models for email spam detection. In total there were 4,601 different emails that were looked at along with 57 different features of the emails. No missing values were found.

Logistic Regression, Naive Bayes, Decision Tree, and Random Forest were trained and evaluated on parsed-out test set. The models were compared using accuracy, precision, recall, F1 score, and confusion matrices to gauge performance.

Overall the random forest model performed the best with ~95% accuracy. The second model was 91% accurate and it was the Logisitic Regression. The Decisions tree and Naive Bayes came in third and fourth place with respective accuracies of 90% and 82%.

Looking at other dimensions of the models, the Random Forest had the best level of false positives (9) and false negatives too (32)